# Actigraphy Pre-extracted Features

In [23]:
import pandas as pd

# Load CSV
features_df = pd.read_csv("hyperaktiv_with_controls/features.csv", sep=";")

# Set participant ID as index
features_df = features_df.set_index("ID")

# This is your X matrix
X = features_df.copy()


In [24]:
print(X.shape)
print(X.isna().sum().sum(), "NaNs total")
print((X.var() == 0).sum(), "zero-variance features")


(116, 787)
696 NaNs total
47 zero-variance features


In [25]:
# Drop columns with any NaNs
X = X.dropna(axis=1)

# Drop zero-variance features
X = X.loc[:, X.var() > 0]

# Optional: reduce precision to save memory
# X = X.astype("float32")


In [26]:
import pandas as pd

# Load patient info
info = pd.read_csv("hyperaktiv_with_controls/patient_info.csv", sep=";")

# Keep only relevant columns
info = info[["ID", "ADHD", "ADD"]]

# Align with X (important!)
info = info.set_index("ID").loc[X.index]

# Create multiclass target
def build_multiclass_label(row):
    if row["ADHD"] == 0:
        return 0  # No ADHD
    elif row["ADHD"] == 1 and row["ADD"] == 0:
        return 1  # ADHD without inattentive subtype
    elif row["ADHD"] == 1 and row["ADD"] == 1:
        return 2  # ADHD with inattentive subtype

y = info.apply(build_multiclass_label, axis=1)

# Convert to integer type
y = y.astype(int)


In [27]:
print(y.value_counts())
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Index aligned:", X.index.equals(y.index))


0    71
2    23
1    22
Name: count, dtype: int64
X shape: (116, 734)
y shape: (116,)
Index aligned: True


In [28]:
from tsfresh.feature_selection import select_features

X_selected = select_features(
    X,
    y,
    fdr_level=0.05
)

print("Original features:", X.shape[1])
print("Selected features:", X_selected.shape[1])


Original features: 734
Selected features: 52


### Split Data

In [29]:
from sklearn.model_selection import train_test_split

# 80/20 split
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=5,
    stratify=y
)

print(f"Train/Val samples: {len(X_train_val)}")
print(f"Test samples: {len(X_test)}")


Train/Val samples: 92
Test samples: 24


### SVM Rough Draft

In [15]:
from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    probability=False,
    random_state=5
)


In [16]:
svm.fit(X_train_scaled, y_train)

from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

y_val_pred = svm.predict(X_val_scaled)

print("Validation balanced accuracy:",
      balanced_accuracy_score(y_val, y_val_pred))

print("Validation macro F1:",
      f1_score(y_val, y_val_pred, average="macro"))

print("\nValidation classification report:")
print(classification_report(y_val, y_val_pred))

print("\nValidation confusion matrix:")
print(confusion_matrix(y_val, y_val_pred))


Validation balanced accuracy: 0.5904761904761905
Validation macro F1: 0.5017948717948718

Validation classification report:
              precision    recall  f1-score   support

           0       0.73      0.57      0.64        14
           1       0.33      0.20      0.25         5
           2       0.44      1.00      0.62         4

    accuracy                           0.57        23
   macro avg       0.50      0.59      0.50        23
weighted avg       0.59      0.57      0.55        23


Validation confusion matrix:
[[8 2 4]
 [3 1 1]
 [0 0 4]]


### SVM Real (stratifiedkfolds, f1 macro)

In [30]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score

# Parameter grid
C_values = [0.1, 1, 10]
gamma_values = [0.1, "scale", 0.001]

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=5)

results = []

for C in C_values:
    for gamma in gamma_values:
        fold_idx = 0

        for train_idx, val_idx in skf.split(X_train_val, y_train_val):
            fold_idx += 1

            X_train, X_val = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
            y_train, y_val = y_train_val.iloc[train_idx], y_train_val.iloc[val_idx]

            pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("svm", SVC(
                    C=C,
                    gamma=gamma,
                    kernel="rbf",
                    class_weight="balanced"
                ))
            ])

            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_val)

            f1 = f1_score(y_val, y_pred, average="macro")

            results.append({
                "C": C,
                "gamma": gamma,
                "fold": fold_idx,
                "f1_macro": f1
            })

results_df = pd.DataFrame(results)
results_df


,C,gamma,fold,f1_macro
0,0.1,0.1,1,0.115942
1,0.1,0.1,2,0.250980
2,0.1,0.1,3,0.121212
3,0.1,0.1,4,0.121212
4,0.1,0.1,5,0.121212
5,0.1,scale,1,0.115942
6,0.1,scale,2,0.468783
7,0.1,scale,3,0.365196
8,0.1,scale,4,0.365196
9,0.1,scale,5,0.235897


In [31]:
summary_df = (
    results_df
    .groupby(["C", "gamma"])["f1_macro"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

summary_df


,C,gamma,mean,std,count
0,0.1,0.001,0.146825,0.060261,5
1,0.1,0.1,0.146112,0.058668,5
2,0.1,scale,0.310203,0.136432,5
3,1.0,0.001,0.352451,0.022289,5
4,1.0,0.1,0.450860,0.091308,5
5,1.0,scale,0.362551,0.068098,5
6,10.0,0.001,0.385057,0.059029,5
7,10.0,0.1,0.405127,0.094030,5
8,10.0,scale,0.461902,0.067110,5


### Best Parameters

In [35]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Final model with best hyperparameters
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        C=10,
        gamma="scale",
        kernel="rbf",
        class_weight="balanced",
        random_state=5
    ))
])

# Train on full 80%
final_model.fit(X_train_val, y_train_val)

# Evaluate on untouched 20% test set
y_test_pred = final_model.predict(X_test)

# Metrics
f1_macro = f1_score(y_test, y_test_pred, average="macro")
balanced_acc = balanced_accuracy_score(y_test, y_test_pred)

print(f"Test Macro F1: {f1_macro:.3f}")
print(f"Test Balanced Accuracy: {balanced_acc:.3f}")

print("\nTest Classification Report:")
print(classification_report(y_test, y_test_pred))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


Test Macro F1: 0.420
Test Balanced Accuracy: 0.417

Test Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.80      0.77        15
           1       0.33      0.25      0.29         4
           2       0.20      0.20      0.20         5

    accuracy                           0.58        24
   macro avg       0.43      0.42      0.42        24
weighted avg       0.57      0.58      0.57        24


Test Confusion Matrix:
[[12  0  3]
 [ 2  1  1]
 [ 2  2  1]]
